
# DAT405 Machine Learning Lab — Lab 1 (Revised)
## Feature Extraction / Selection / Dimensionality Reduction — Decision-Contingent Version

**Dataset file:** `lab1_section_dataset.csv`  
**SHA-256 checksum:** `560cc26b031ddfeed16025b21d8be52fe42fd0dd0172483894ba53eb8f3a7bcf`

### Rules (graded)
- You may use `pandas`, `numpy`, `matplotlib`, and **models** from `sklearn`.
- You may **not** use built-in feature selection shortcuts:
  - `sklearn.feature_selection.VarianceThreshold`
  - `sklearn.feature_selection.SelectKBest`, `SelectPercentile`
  - `mutual_info_classif`, `mutual_info_regression`
  - `DataFrame.corr()`
- You must compute **variance**, **Pearson correlation**, and an **approximate mutual information** score manually.
- You must keep all intermediate outputs (printed values and plots).

### What makes this lab different
The dataset has:
- renamed (scrambled) feature columns
- redundant/highly-correlated features
- quasi-constant features
- missing values
- **one leakage-like feature** and **one spurious feature**

Your job is to build a feature pipeline that is **robust** and **stable**, not just high-scoring once.



## 0) Setup
Run this cell once.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier

np.set_printoptions(suppress=True, precision=4)
pd.set_option("display.max_columns", 50)



## 1) Load the dataset and verify checksum

- Load `lab1_section_dataset.csv`.
- Verify its SHA-256 equals the provided checksum.


In [ ]:

import hashlib

DATA_PATH = "lab1_section_dataset.csv"

# Load
df = pd.read_csv(DATA_PATH)
df.head()

# Verify checksum
with open(DATA_PATH, "rb") as f:
    sha = hashlib.sha256(f.read()).hexdigest()
print("SHA-256:", sha)
print("Matches expected:", sha == "560cc26b031ddfeed16025b21d8be52fe42fd0dd0172483894ba53eb8f3a7bcf")

print("Shape:", df.shape)
print("Target balance:", df["target"].mean())



## 2) Split once, then evaluate across seeds

You must:
- keep one fixed train/test split (use `random_state=42`)
- evaluate model stability across 5 random seeds **only inside the model**


In [ ]:

X = df.drop(columns=["target"])
y = df["target"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)



## 3) Baseline model (no feature selection)

Report:
- ROC-AUC on train and test
- mean and std of test ROC-AUC across 5 seeds (change only `random_state` in the model)


In [ ]:

def eval_rf(Xtr, Xte, ytr, yte, seeds=(0,1,2,3,4)):
    train_scores = []
    test_scores = []
    for s in seeds:
        rf = RandomForestClassifier(
            n_estimators=250,
            max_depth=5,
            random_state=s,
            n_jobs=-1
        )
        rf.fit(Xtr, ytr)
        p_tr = rf.predict_proba(Xtr)[:,1]
        p_te = rf.predict_proba(Xte)[:,1]
        train_scores.append(roc_auc_score(ytr, p_tr))
        test_scores.append(roc_auc_score(yte, p_te))
    return np.array(train_scores), np.array(test_scores)

tr, te = eval_rf(X_train.fillna(X_train.median(numeric_only=True)),
                 X_test.fillna(X_train.median(numeric_only=True)),
                 y_train, y_test)

print("Train AUCs:", tr)
print("Test  AUCs:", te)
print("Test mean±std:", te.mean(), te.std())



## 4) Manual variance filter (constant + quasi-constant)

Implement:
- `manual_variance(series)` using the definition of variance
- remove features with variance <= `eps`
- choose **two** eps values:
  1) removes only constant-like features
  2) removes quasi-constant features too

Deliverables:
- list removed features for each eps
- stability report (mean±std test AUC across seeds) after removal


In [ ]:

def manual_variance(x: pd.Series) -> float:
    # x may include NaN; ignore NaN
    v = x.dropna().to_numpy(dtype=float)
    mu = v.mean()
    return ((v - mu)**2).mean()

def variance_filter(X: pd.DataFrame, eps: float):
    vars_ = {c: manual_variance(X[c]) for c in X.columns}
    keep = [c for c, vv in vars_.items() if vv > eps]
    drop = [c for c in X.columns if c not in keep]
    return keep, drop, vars_

# TODO: set eps1 and eps2
eps1 = 0.0
eps2 = 0.005

keep1, drop1, vars1 = variance_filter(X_train, eps1)
keep2, drop2, vars2 = variance_filter(X_train, eps2)

print("eps1 dropped:", drop1)
print("eps2 dropped:", drop2)
print("Counts (keep/drop) eps2:", len(keep2), len(drop2))

# Evaluate eps2 pipeline
med = X_train[keep2].median(numeric_only=True)
tr2, te2 = eval_rf(X_train[keep2].fillna(med), X_test[keep2].fillna(med), y_train, y_test)
print("After variance filter (eps2) Test mean±std:", te2.mean(), te2.std())



## 5) Manual Pearson correlation pruning (no `DataFrame.corr()`)

Implement Pearson correlation from definition:
- For each pair (i, j), compute |corr|
- If |corr| > threshold, drop **one** feature (your policy must be stated and consistent)

Deliverables:
- your policy (markdown)
- number of removed features
- stability report (mean±std test AUC across seeds)

Important:
- correlation must be computed on **train** only
- handle NaNs (drop NaN pairs)



### Correlation pruning policy (write yours)
Example policy: "When a pair is highly correlated, drop the feature with lower absolute correlation with the target (measured later by MI), otherwise drop the second feature by column order."

Replace this text with your policy.


In [ ]:

def manual_pearson(a: pd.Series, b: pd.Series) -> float:
    # align and drop NaNs pairwise
    tmp = pd.concat([a, b], axis=1).dropna()
    x = tmp.iloc[:,0].to_numpy(dtype=float)
    y = tmp.iloc[:,1].to_numpy(dtype=float)
    if len(x) < 3:
        return 0.0
    xm = x.mean(); ym = y.mean()
    num = ((x-xm)*(y-ym)).sum()
    den = np.sqrt(((x-xm)**2).sum()) * np.sqrt(((y-ym)**2).sum())
    if den == 0:
        return 0.0
    return float(num/den)

def correlation_prune(X: pd.DataFrame, threshold: float):
    cols = list(X.columns)
    drop = set()
    # simple default policy: drop later column
    for i in range(len(cols)):
        if cols[i] in drop:
            continue
        for j in range(i+1, len(cols)):
            if cols[j] in drop:
                continue
            c = manual_pearson(X[cols[i]], X[cols[j]])
            if abs(c) > threshold:
                drop.add(cols[j])
    keep = [c for c in cols if c not in drop]
    return keep, sorted(drop)

# Start from variance-filtered set (eps2 by default)
start_cols = keep2
keep_corr, drop_corr = correlation_prune(X_train[start_cols], threshold=0.85)

print("Dropped by correlation:", len(drop_corr))
print(drop_corr[:30])

med = X_train[keep_corr].median(numeric_only=True)
tr3, te3 = eval_rf(X_train[keep_corr].fillna(med), X_test[keep_corr].fillna(med), y_train, y_test)
print("After corr prune Test mean±std:", te3.mean(), te3.std())



## 6) Approximate Mutual Information (manual; discretization)

Compute an **approximate MI** score I(X; y) by:
1) discretizing each continuous feature into `k` bins (quantiles or equal-width)
2) computing empirical probabilities P(x_bin, y), P(x_bin), P(y)
3) summing: MI = Σ P(x,y) log( P(x,y) / (P(x)P(y)) )

Deliverables:
- your MI implementation
- top-10 features by MI (train only)
- stability report using top-k features (k=5, 10, 15)


In [ ]:

def discretize_quantile(x: pd.Series, k: int) -> pd.Series:
    # returns bin ids 0..k-1; NaNs stay NaN
    return pd.qcut(x, q=k, labels=False, duplicates="drop")

def approx_mi_discrete(xbins: pd.Series, y: pd.Series) -> float:
    tmp = pd.concat([xbins, y], axis=1).dropna()
    xb = tmp.iloc[:,0].astype(int).to_numpy()
    yy = tmp.iloc[:,1].astype(int).to_numpy()
    n = len(tmp)
    if n == 0:
        return 0.0
    # joint counts
    joint = {}
    cx = {}
    cy = {}
    for a, b in zip(xb, yy):
        joint[(a,b)] = joint.get((a,b), 0) + 1
        cx[a] = cx.get(a, 0) + 1
        cy[b] = cy.get(b, 0) + 1
    mi = 0.0
    for (a,b), c in joint.items():
        pxy = c / n
        px = cx[a] / n
        py = cy[b] / n
        mi += pxy * np.log(pxy / (px*py + 1e-12) + 1e-12)
    return float(mi)

def mi_scores(X: pd.DataFrame, y: pd.Series, k_bins: int = 10):
    scores = {}
    for c in X.columns:
        xb = discretize_quantile(X[c], k_bins)
        scores[c] = approx_mi_discrete(xb, y)
    return pd.Series(scores).sort_values(ascending=False)

mi = mi_scores(X_train[keep_corr], y_train, k_bins=10)
print(mi.head(15))

top10 = list(mi.head(10).index)
print("Top-10:", top10)

def eval_topk(k):
    cols = list(mi.head(k).index)
    med = X_train[cols].median(numeric_only=True)
    tr, te = eval_rf(X_train[cols].fillna(med), X_test[cols].fillna(med), y_train, y_test)
    return te.mean(), te.std(), cols

for k in [5, 10, 15]:
    m, s, cols = eval_topk(k)
    print(f"k={k}: test mean±std = {m:.4f} ± {s:.4f}")



## 7) The required trap: identify the harmful method

One of your feature-selection steps should **hurt** test AUC or stability.

Deliverables:
- identify which step hurt performance
- show the evidence (numbers)
- explain why (overfitting, leakage reliance, instability, thresholding, etc.)



## 8) Irreversible design choice (dimensionality reduction)

Choose **one** path and stick to it:
- Path A: PCA on standardized features
- Path B: TruncatedSVD on standardized features
- Path C: No DR; only selection

Write your justification **before** implementing.
Then implement and report stability metrics.



### My choice (write here)
- Selected path: (A/B/C)
- Why this path is appropriate for this dataset:
- What failure mode it avoids:


In [ ]:

# TODO: implement ONLY your chosen path
# If you choose PCA/SVD, standardize using train mean/std and apply to test (no leakage).




## 9) Submission checklist

Your notebook must include:
- the verified checksum output
- baseline AUC mean±std (5 seeds)
- variance filter implementation + results
- correlation implementation + results
- MI implementation + results
- the “harmful method” identification and explanation
- DR path justification + results
